# 01 - Configuración del Entorno y Exploración Inicial

**Modelo Cognitivo Artificial para la Replicación de la Actividad Neurofisiológica de Percepción de Profundidad**

- **Autor:** Jesús Goenaga Peña
- **Programa:** Doctorado en Ciencias Cognitivas - Universidad Autónoma de Manizales
- **Tutor:** Luis Fernando Castillo Ossa

---

Este notebook configura el entorno de trabajo en Google Colab, verifica la disponibilidad de GPU,
instala las dependencias necesarias y genera el reporte de entorno para incluir en la tesis.

## 1. Verificación de GPU

**IMPORTANTE:** Antes de ejecutar esta celda, asegúrate de que Colab tenga GPU activada:
1. Ve a **Entorno de ejecución** (Runtime) en el menú superior
2. Selecciona **Cambiar tipo de entorno de ejecución** (Change runtime type)
3. En **Acelerador de hardware** selecciona **T4 GPU**
4. Dale **Guardar**

In [ ]:
# Verificar que tenemos GPU disponible
import torch

print('=' * 60)
print('VERIFICACIÓN DE GPU')
print('=' * 60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU detectada: {gpu_name}')
    print(f'Memoria GPU: {gpu_mem:.1f} GB')
    print(f'CUDA versión: {torch.version.cuda}')
    print(f'\nEstado: LISTO para entrenamiento con GPU')
else:
    print('ADVERTENCIA: No se detectó GPU.')
    print('Ve a Runtime > Change runtime type > T4 GPU')
    print('Luego reinicia este notebook.')

VERIFICACIÓN DE GPU
GPU detectada: Tesla T4
Memoria GPU: 15.6 GB
CUDA versión: 12.8

Estado: LISTO para entrenamiento con GPU


## 2. Instalación de Dependencias

In [1]:
# Instalar dependencias que no vienen por defecto en Colab
!pip install -q grad-cam captum scikit-image pyyaml tabulate colorama 2>/dev/null
!pip install -q numpy==2.0.2 2>/dev/null

print('✓ Dependencias adicionales instaladas correctamente.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 45.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 27.4 MB/s eta 0:00:00
✓ Dependencias adicionales instaladas correctamente.


## 3. Montaje de Google Drive

Google Drive se usa para guardar de forma persistente:
- Checkpoints del modelo (pesos entrenados)
- Resultados y métricas
- Visualizaciones y mapas Grad-CAM
- Datos procesados

In [ ]:
from google.colab import drive
import os

# Montar Drive
drive.mount('/content/drive')

# Crear estructura del proyecto en Drive
PROJECT_ROOT = '/content/drive/MyDrive/cognitive-depth-model'

dirs_to_create = [
    'checkpoints',
    'results/metrics',
    'results/visualizations',
    'results/grad_cam',
    'data/raw/kitti',
    'data/raw/illusions',
    'data/processed',
    'logs',
]

for d in dirs_to_create:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)

print(f'Proyecto configurado en: {PROJECT_ROOT}')
print(f'\nEstructura creada:')
for d in dirs_to_create:
    print(f'  {PROJECT_ROOT}/{d}/')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Proyecto configurado en: /content/drive/MyDrive/cognitive-depth-model

Estructura creada:
  /content/drive/MyDrive/cognitive-depth-model/checkpoints/
  /content/drive/MyDrive/cognitive-depth-model/results/metrics/
  /content/drive/MyDrive/cognitive-depth-model/results/visualizations/
  /content/drive/MyDrive/cognitive-depth-model/results/grad_cam/
  /content/drive/MyDrive/cognitive-depth-model/data/raw/kitti/
  /content/drive/MyDrive/cognitive-depth-model/data/raw/illusions/
  /content/drive/MyDrive/cognitive-depth-model/data/processed/
  /content/drive/MyDrive/cognitive-depth-model/logs/


## 4. Clonar Repositorio de GitHub

In [ ]:
# Clonar el repositorio del proyecto
REPO_URL = 'https://github.com/yisusgoenaga/cognitive-depth-model.git'
REPO_DIR = '/content/cognitive-depth-model'

if os.path.exists(REPO_DIR):
    print(f'El repositorio ya existe en {REPO_DIR}')
    print('Actualizando con git pull...')
    !cd {REPO_DIR} && git pull
else:
    print(f'Clonando repositorio desde {REPO_URL}...')
    !git clone {REPO_URL} {REPO_DIR}

# Agregar el src al path de Python
import sys
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

print(f'\nRepositorio listo en: {REPO_DIR}')

El repositorio ya existe en /content/cognitive-depth-model
Actualizando con git pull...
Already up to date.

Repositorio listo en: /content/cognitive-depth-model


## 5. Configuración de Reproducibilidad

In [ ]:
import torch
import numpy as np
import random

SEED = 42

def set_seed(seed=SEED):
    """Fija todas las semillas para reproducibilidad completa."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f'Semillas de reproducibilidad configuradas (seed={seed})')

set_seed()

Semillas de reproducibilidad configuradas (seed=42)


## 6. Reporte del Entorno para la Tesis

Esta celda genera el reporte completo de versiones que se debe incluir
en la sección metodológica de la tesis (Sección 8.5.5).

In [7]:
import os
import platform
from datetime import datetime
import torch
import torchvision
import numpy as np
import scipy
import cv2
import sklearn
import matplotlib
import pandas as pd
import seaborn

print('=' * 65)
print('REPORTE DE ENTORNO COMPUTACIONAL PARA LA TESIS')
print(f'Fecha de generación: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 65)
print()
print('--- Entorno computacional del proyecto ---')
print()
print(f'  Python:        {platform.python_version()}')
print(f'  PyTorch:       {torch.__version__}')
print(f'  TorchVision:   {torchvision.__version__}')
print(f'  NumPy:         {np.__version__}')
print(f'  SciPy:         {scipy.__version__}')
print(f'  OpenCV:        {cv2.__version__}')
print(f'  scikit-learn:  {sklearn.__version__}')
print(f'  Matplotlib:    {matplotlib.__version__}')
print(f'  Pandas:        {pd.__version__}')
print(f'  Seaborn:       {seaborn.__version__}')
print()
if torch.cuda.is_available():
    print(f'  CUDA:          {torch.version.cuda}')
    print(f'  GPU:           {torch.cuda.get_device_name(0)}')
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  Memoria GPU:   {gpu_mem:.1f} GB')
print()
print(f'  Plataforma:    Google Colaboratory')
print(f'  SO:            {platform.system()} {platform.release()}')
print()
print('=' * 65)
print()



REPORTE DE ENTORNO COMPUTACIONAL PARA LA TESIS
Fecha de generación: 2026-04-23 14:52:34

--- Entorno computacional del proyecto ---

  Python:        3.12.13
  PyTorch:       2.10.0+cu128
  TorchVision:   0.25.0+cu128
  NumPy:         2.0.2
  SciPy:         1.16.3
  OpenCV:        4.13.0
  scikit-learn:  1.6.1
  Matplotlib:    3.10.0
  Pandas:        2.2.2
  Seaborn:       0.13.2

  CUDA:          12.8
  GPU:           Tesla T4
  Memoria GPU:   15.6 GB

  Plataforma:    Google Colaboratory
  SO:            Linux 6.6.113+




## 7. Test Rápido de PyTorch + GPU

Verificamos que PyTorch puede crear tensores en la GPU y realizar operaciones básicas.

In [ ]:
# Definir dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo activo: {device}')

# Test: crear un tensor y moverlo a GPU
x = torch.randn(3, 256, 512).to(device)  # Simula una imagen RGB
print(f'\nTensor de prueba creado en {x.device}:')
print(f'  Forma: {x.shape}  (canales, alto, ancho)')
print(f'  Tipo:  {x.dtype}')
print(f'  Min:   {x.min().item():.4f}')
print(f'  Max:   {x.max().item():.4f}')
print(f'  Media: {x.mean().item():.4f}')

# Test: operación convolucional básica en GPU
conv = torch.nn.Conv2d(3, 64, kernel_size=3, padding=1).to(device)
y = conv(x.unsqueeze(0))  # Agregar dimensión de batch
print(f'\nConvolución de prueba ejecutada en GPU:')
print(f'  Entrada:  {x.unsqueeze(0).shape}')
print(f'  Salida:   {y.shape}')
print(f'  Filtros:  64 filtros de 3x3')

print(f'\nTodo funciona correctamente en {device}.')

Dispositivo activo: cuda

Tensor de prueba creado en cuda:0:
  Forma: torch.Size([3, 256, 512])  (canales, alto, ancho)
  Tipo:  torch.float32
  Min:   -4.5905
  Max:   4.6291
  Media: -0.0014

Convolución de prueba ejecutada en GPU:
  Entrada:  torch.Size([1, 3, 256, 512])
  Salida:   torch.Size([1, 64, 256, 512])
  Filtros:  64 filtros de 3x3

Todo funciona correctamente en cuda.


## 8. Resumen y Próximos Pasos

Si todas las celdas anteriores se ejecutaron sin errores:

- GPU configurada y verificada
- Dependencias instaladas
- Google Drive montado con estructura del proyecto
- Repositorio clonado
- Reproducibilidad configurada (seed=42)
- Reporte de entorno generado para la tesis

**Siguiente notebook:** `02_data_download_and_exploration.ipynb`
- Descarga de datasets (KITTI Scene Flow 2015 + 3D Visual Illusion)
- Implementación de las Fases 1-6 del modelo (pipeline retiniano)

In [ ]:
print('=' * 60)
print('NOTEBOOK 01 COMPLETADO EXITOSAMENTE')
print('=' * 60)
print()
print('Resumen:')
print(f'  Dispositivo:  {device}')
print(f'  Proyecto en:  {PROJECT_ROOT}')
print(f'  Repositorio:  {REPO_DIR}')
print(f'  Seed:         {SEED}')
print()
print('Siguiente paso: Abrir 02_data_download_and_exploration.ipynb')

NOTEBOOK 01 COMPLETADO EXITOSAMENTE

Resumen:
  Dispositivo:  cuda
  Proyecto en:  /content/drive/MyDrive/cognitive-depth-model
  Repositorio:  /content/cognitive-depth-model
  Seed:         42

Siguiente paso: Abrir 02_data_download_and_exploration.ipynb
